In [1]:
# ==============================================================
# 09_ensemble_boosting_regressors.ipynb
# --------------------------------------------------------------
# Blends multiple advanced regressors (LightGBM, XGBoost, CatBoost, etc.)
# and automatically selects top 3 per stock for weighted blending
# ==============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

# --- External libraries
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ==============================================================
# Configuration
# ==============================================================
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_enhanced_model_ready.csv",
    "TCS": data_dir / "tcs_enhanced_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_enhanced_model_ready.csv",
}

# ==============================================================
# Models
# ==============================================================
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=150, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42, n_estimators=200),
    "XGBoost": XGBRegressor(random_state=42, n_estimators=200, learning_rate=0.05),
    "CatBoost": CatBoostRegressor(verbose=0, random_state=42, iterations=200, learning_rate=0.05),
}

# ==============================================================
# Helper
# ==============================================================
def evaluate(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

# ==============================================================
# Main Execution
# ==============================================================
all_results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")
    if not path.exists():
        print(f"⚠️ Missing: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded: {df.shape}")

    target_col = "Target_Reg"
    if target_col not in df.columns:
        print(f"⚠️ Skipping {ticker} — no {target_col}")
        continue

    df = df.replace([np.inf, -np.inf], np.nan)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df = df[numeric_cols].dropna(axis=1, how="all")

    if target_col not in df.columns:
        continue

    imputer = SimpleImputer(strategy="mean")
    df[df.columns] = imputer.fit_transform(df)

    y = df[target_col]
    X = df.drop(columns=[target_col], errors="ignore")

    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    model_scores = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        metrics = evaluate(y_test, preds)
        metrics.update({"Model": name, "Ticker": ticker})
        all_results.append(metrics)
        model_scores.append((name, metrics["R2"], preds))

        print(f"  → {name}: RMSE={metrics['RMSE']:.3f}, R2={metrics['R2']:.3f}")

    # --- Select top 3 by R² ---
    top3 = sorted(model_scores, key=lambda x: x[1], reverse=True)[:3]

    # --- Simple Blend ---
    blend_simple = np.mean([p for _, _, p in top3], axis=0)
    blend_metrics = evaluate(y_test, blend_simple)
    blend_metrics.update({"Model": "Blend_Simple", "Ticker": ticker})
    all_results.append(blend_metrics)
    print(f"  ✅ Blend_Simple: RMSE={blend_metrics['RMSE']:.3f}, R2={blend_metrics['R2']:.3f}")

    # --- Weighted Blend ---
    weights = np.array([max(r2, 0) for _, r2, _ in top3])
    weights = weights / weights.sum() if weights.sum() != 0 else np.ones(3)/3
    blend_weighted = np.average([p for _, _, p in top3], axis=0, weights=weights)
    blend_w_metrics = evaluate(y_test, blend_weighted)
    blend_w_metrics.update({"Model": "Blend_Weighted", "Ticker": ticker})
    all_results.append(blend_w_metrics)
    print(f"  ✅ Blend_Weighted: RMSE={blend_w_metrics['RMSE']:.3f}, R2={blend_w_metrics['R2']:.3f}")

# ==============================================================
# Save Results
# ==============================================================
results_df = pd.DataFrame(all_results)
save_path = results_dir / "ensemble_boosting_regressors_results.csv"
results_df.to_csv(save_path, index=False)

print(f"\n✅ Advanced Ensemble Blending completed. Results saved to: {save_path}")
display(results_df.sort_values(["Ticker", "R2"], ascending=[True, False]))



=== Processing RELIANCE ===
  Loaded: (1460, 25)
  → Linear: RMSE=17.585, R2=0.977
  → Ridge: RMSE=19.448, R2=0.972
  → Lasso: RMSE=17.960, R2=0.976
  → DecisionTree: RMSE=197.705, R2=-1.890
  → RandomForest: RMSE=180.093, R2=-1.398
  → GradientBoosting: RMSE=172.666, R2=-1.205
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000636 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5345
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 22
[LightGBM] [Info] Start training from score 934.557691
  → LightGBM: RMSE=177.040, R2=-1.318
  → XGBoost: RMSE=180.381, R2=-1.406
  → CatBoost: RMSE=182.207, R2=-1.455
  ✅ Blend_Simple: RMSE=18.122, R2=0.976
  ✅ Blend_Weighted: RMSE=18.121, R2=0.976

=== Processing TCS ===
  Loaded: (1460, 25)
  → Linear: RMSE=52.021, R2=0.971
  → Ridge: RMSE=59.556, R2=0.962
  → Lasso: RMSE=55.021, R2=0.967
  → DecisionTree: RMSE=652.852, R2=-3

,RMSE,MAE,R2,Model,Ticker
22,9.566634,6.859892,0.977250,Linear,HDFCBANK
32,10.064616,7.243581,0.974820,Blend_Weighted,HDFCBANK
31,10.067325,7.246568,0.974807,Blend_Simple,HDFCBANK
24,10.071847,7.172402,0.974784,Lasso,HDFCBANK
23,11.397678,8.551675,0.967708,Ridge,HDFCBANK
28,33.695680,21.810379,0.717768,LightGBM,HDFCBANK
27,33.873945,22.140068,0.714774,GradientBoosting,HDFCBANK
26,35.259180,23.609869,0.690969,RandomForest,HDFCBANK
29,35.302867,23.316934,0.690202,XGBoost,HDFCBANK
30,37.385540,25.563013,0.652571,CatBoost,HDFCBANK
